# pMTG Spatial Variant Validation

This notebook implements the pMTG variant-validation analyses requested for the manuscript revision:

1. Load participant-level unthresholded spatial-correlation maps.
2. Threshold each participant at the bottom 5%, 10%, and 15% of spatial-correlation values.
3. Cluster subject-level thresholded maps with a 30 mm2 minimum-contiguous-area criterion.
4. Generate cPFM (n = 10) density maps for the 5%, 10%, and 15% thresholds.
5. Examine degree of spatial overlap between variant density maps (at the lowest decile threshold) for typically developing subjects versus subjects with a neurodevelopmental disorder diagnosis
6. Run whole-cohort jackknife analyses to determine sensitivity of pMTG region definition to removal of single subjects using leave-one-out Dice summaries


## Assumptions

- The reported cohort contains `MSCPI05`, `MSCPI07`, `MSCPI08`, `MSCPI10`, `MSCPI12`, `MSCPI14`, `MSCPI15`, `MSCPI17`, `MSCPI18`, and `MSCPI19`.
- `MSCPI14`, `MSCPI15`, and `MSCPI18` have neurodevelopmental diagnoses
- Full-cohort consensus pMTG regions of high inter-individual variabilty require 5/10 participants, diagnosis-excluded regions require 4/7 participants, and jackknife consensus requires 4/9 participants.
- Jackknife sensitivity is summarized from leave-one-subject-out Dice values across the full reported cohort, not from the diagnosis-excluded subset.


In [ ]:
import re

import subprocess

from pathlib import Path

import matplotlib.pyplot as plt

import nibabel as nib

import numpy as np

import pandas as pd

import seaborn as sns

# Use a clean plotting style for all summary figures.
sns.set_theme(style="whitegrid", context="notebook")


In [ ]:
# Store the Connectome Workbench executable used for surface clustering.
WORKBENCH = Path("/Applications/workbench/bin_macosx64/wb_command")

# Store the folder containing the unthresholded participant spatial-correlation maps.
VARIANT_DIR = Path(
    "/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/Variants"
)

# Store the left and right HCP 32k midthickness surfaces required by Workbench.
SURFACE_PATHS = {
    "left": Path(
        "/Users/emk/Documents/Documents - Ron Weasley V/Research/Data/Imaging/"
        "Surfaces/HCP1200/S1200.L.midthickness_MSMAll.32k_fs_LR.surf.gii"
    ),
    "right": Path(
        "/Users/emk/Documents/Documents - Ron Weasley V/Research/Data/Imaging/"
        "Surfaces/HCP1200/S1200.R.midthickness_MSMAll.32k_fs_LR.surf.gii"
    ),
}

# Store the folder where all generated maps, tables, and figures are written.
OUTPUT_DIR = Path.cwd() / "variant_spatial_validation_results"

# Store the folder where thresholded subject maps are written.
SUBJECT_OUTPUT_DIR = OUTPUT_DIR / "subject_variant_maps"

# Store the folder where group consensus and pMTG maps are written.
GROUP_OUTPUT_DIR = OUTPUT_DIR / "group_variant_maps"

# Define the participants retained in the diagnosis-excluded analysis.
TD_SUBJECTS = [
    "MSCPI05",
    "MSCPI07",
    "MSCPI08",
    "MSCPI10",
    "MSCPI12",
    "MSCPI17",
    "MSCPI19",
]

# Define the participants with reported neurodevelopmental diagnoses.
ND_SUBJECTS = ["MSCPI14", "MSCPI15", "MSCPI18"]

# Define the full reported cohort in the order used throughout the notebook.
COHORT_SUBJECTS = TD_SUBJECTS + ND_SUBJECTS

# Define the spatial-correlation lower-tail thresholds used for group-map output.
VARIANT_PERCENTILES = [5.0, 10.0, 15.0]

# Define the threshold used for diagnosis-exclusion and jackknife analyses.
JACKKNIFE_PERCENTILE = 10.0

# Define 4/9 jackknife consensus rule.
JACKKNIFE_MIN_SUBJECT_COUNT = 4

# Define the subject-level minimum contiguous surface area in square millimeters.
SUBJECT_MIN_CLUSTER_AREA_MM2 = 30.0

# Define the group-level minimum contiguous surface area in square millimeters (40 and 30 give the same result when defining variants as the lowest decile of correlations).
GROUP_MIN_CLUSTER_AREA_MM2 = 40.0

# Define the left and right pMTG target coordinates used to rank candidate clusters (for automated detection).
PMTG_TARGET_COORDINATES = {
    "left": np.asarray([-58.0, -54.0, 3.0]),
    "right": np.asarray([58.0, -49.0, 5.0]),
}

# Create the top-level output folder if it does not already exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create the subject-map output folder if it does not already exist.
SUBJECT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create the group-map output folder if it does not already exist.
GROUP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)




## Input Spatial-Correlation Maps

The code below indexes every available `spatialCorrMap` file, verifies that every reported participant has a map, and records which extra maps are present but not analyzed.


In [ ]:
# Compile the filename pattern used to parse participant IDs.
SPATIAL_CORRELATION_PATTERN = re.compile(
    r"(?P<subject>MSCPI\d+)spatialCorrMap\.dtseries\.nii$",
    flags=re.IGNORECASE,
)


def parse_spatial_correlation_filename(path):
    """Return the participant ID encoded in a spatial-correlation filename."""

    # Convert the input to a Path so filename parsing is consistent.
    path = Path(path)

    # Match the expected spatial-correlation filename pattern.
    match = SPATIAL_CORRELATION_PATTERN.search(path.name)

    # Stop immediately if the filename does not contain a participant ID.

    # Return the participant ID in a consistent uppercase format.
    return match.group("subject").upper()


def index_spatial_correlation_files(paths):
    """Return a dictionary mapping participant IDs to spatial-correlation maps."""

    index = {}

    # Visit each discovered spatial-correlation map.
    for path in paths:

        # Parse the participant ID from the current filename.
        subject = parse_spatial_correlation_filename(path)

        # Stop if two files claim to be the same participant.

        # Store the map path under the parsed participant ID.
        index[subject] = Path(path)

    # Return the completed participant-to-path dictionary.
    return index


# Find every unthresholded participant spatial-correlation map in the input folder.
spatial_correlation_paths = sorted(VARIANT_DIR.glob("*spatialCorrMap.dtseries.nii"))

# Index the discovered maps by participant ID.
spatial_correlation_index = index_spatial_correlation_files(
    spatial_correlation_paths
)

# Identify reported participants that are missing spatial-correlation maps.
missing_subjects = [
    subject for subject in COHORT_SUBJECTS
    if subject not in spatial_correlation_index
]

# Stop if any reported participant map is missing.

# Stop if the configured Workbench executable is absent.

# Stop if either configured surface file is absent.
# Build an audit table describing all spatial-correlation maps found.
input_audit_df = pd.DataFrame(
    [
        {
            "subject": subject,
            "included_in_reported_cohort": subject in COHORT_SUBJECTS,
            "spatial_correlation_path": str(path),
        }
        for subject, path in sorted(spatial_correlation_index.items())
    ]
)

# Display the input audit table in the notebook.
display(input_audit_df)

# Save the input audit table for reproducibility.
input_audit_df.to_csv(OUTPUT_DIR / 'spatial_correlation_input_audit.csv', index=False)


## CIFTI and Workbench Helpers

These helper functions load CIFTI vectors, save dense scalar maps, run Workbench clustering, summarize cluster locations, select pMTG labels, and compute Dice coefficient.


In [ ]:
# Load one cohort map to define the shared CIFTI brain-model axis.
template_image = nib.load(spatial_correlation_index[COHORT_SUBJECTS[0]])

# Extract the grayordinate axis from the template map.
brain_axis = template_image.header.get_axis(1)

# Store the expected number of grayordinates for all maps in this analysis.
n_grayordinates = brain_axis.size

hemisphere_models = {}

# Iterate over every brain structure present in the template CIFTI axis.
for structure_name, structure_slice, brain_model in brain_axis.iter_structures():

    # Convert the structure name to text for simple hemisphere matching.
    structure_text = str(structure_name)

    # Assign the Workbench structure to the left hemisphere when appropriate.
    if "CORTEX_LEFT" in structure_text:
        hemisphere = "left"

    # Assign the Workbench structure to the right hemisphere when appropriate.
    elif "CORTEX_RIGHT" in structure_text:
        hemisphere = "right"

    # Skip non-cortical structures if any are present.
    else:
        continue

    # Store the slice and full-surface vertex indices for this hemisphere.
    hemisphere_models[hemisphere] = {
        "slice": structure_slice,
        "vertex_indices": np.asarray(brain_model.vertex, dtype=int),
    }

# Load full-surface coordinates used to summarize cluster centroids.
surface_coordinates = {
    hemisphere: np.asarray(nib.load(path).darrays[0].data)
    for hemisphere, path in SURFACE_PATHS.items()
}


def load_cifti_vector(path):
    """Load a single-map CIFTI file as a one-dimensional grayordinate vector."""

    # Load the CIFTI file and collapse singleton dimensions.
    values = np.asarray(nib.load(path).dataobj).squeeze()

    # Stop if the map does not match the template grayordinate axis length.

    # Return the validated grayordinate vector.
    return values


def save_scalar_cifti(values, path, map_name):
    """Save one grayordinate vector as a dense scalar CIFTI map."""

    # Create a scalar axis containing the requested map name.
    scalar_axis = nib.cifti2.cifti2_axes.ScalarAxis([map_name])

    # Combine the scalar axis with the template brain-model axis.
    scalar_header = nib.Cifti2Header.from_axes((scalar_axis, brain_axis))

    # Create a dense scalar image using float32 data for Workbench compatibility.
    output_image = nib.Cifti2Image(
        np.asarray(values, dtype=np.float32).reshape(1, -1),
        header=scalar_header,
    )

    # Mark the NIfTI intent as dense scalar so downstream tools read it correctly.
    output_image.nifti_header.set_intent("ConnDenseScalar")

    # Write the dense scalar CIFTI map to disk.
    nib.save(output_image, path)


def run_find_clusters(input_path, value_threshold, minimum_area_mm2, output_path, less_than=False):
    """Run Workbench `cifti-find-clusters` on cortical surfaces."""

    # Build the shared Workbench command arguments.
    command = [
        str(WORKBENCH),
        "-cifti-find-clusters",
        str(input_path),
        str(value_threshold),
        str(minimum_area_mm2),
        "0",
        "0",
        "COLUMN",
        str(output_path),
    ]

    # Add lower-tail thresholding when requested for subject spatial-correlation maps.
    if less_than:
        command.append("-less-than")

    # Add the left cortical surface required for surface-area clustering.
    command.extend(["-left-surface", str(SURFACE_PATHS["left"])])

    # Add the right cortical surface required for surface-area clustering.
    command.extend(["-right-surface", str(SURFACE_PATHS["right"])])

    # Run Workbench and raise an error if clustering fails.
    subprocess.run(command, check=True, capture_output=True, text=True)


def build_group_clusters(subject_cluster_paths, analysis_name, min_subjects):
    """Create density and consensus-cluster maps from subject-level maps."""

    # Load every subject cluster map included in this analysis.
    subject_cluster_values = [
        load_cifti_vector(path) for path in subject_cluster_paths
    ]

    # Binarize each subject map and sum across participants to make a density map.
    density = np.asarray(subject_cluster_values).astype(bool).sum(axis=0, dtype=np.int16)

    # Store the density map path.
    density_path = GROUP_OUTPUT_DIR / f"{analysis_name}_density.dscalar.nii"

    # Store the consensus cluster map path.
    cluster_path = GROUP_OUTPUT_DIR / f"{analysis_name}_clusters.dscalar.nii"

    # Save the density map for visual inspection.
    save_scalar_cifti(density, density_path, f"{analysis_name}_density")

    # Cluster vertices present in at least `min_subjects` participants.
    run_find_clusters(
        density_path,
        min_subjects - 1,
        GROUP_MIN_CLUSTER_AREA_MM2,
        cluster_path,
    )

    # Return the generated paths, values, and consensus count.
    return {
        "density": density,
        "density_path": density_path,
        "cluster_path": cluster_path,
        "cluster_values": load_cifti_vector(cluster_path),
        "min_subjects": min_subjects,
    }


def summarize_clusters(cluster_values, analysis_name):
    """Summarize Workbench cluster labels and centroid distances from pMTG targets."""

    rows = []

    # Summarize clusters separately in each hemisphere.
    for hemisphere in ("left", "right"):

        # Retrieve this hemisphere's CIFTI slice and vertex index mapping.
        model = hemisphere_models[hemisphere]

        # Extract this hemisphere's cluster values from the CIFTI vector.
        hemisphere_values = cluster_values[model["slice"]].astype(int)

        # Retrieve full-surface vertex indices for the hemisphere CIFTI slice.
        full_vertex_indices = model["vertex_indices"]

        # Visit each nonzero Workbench cluster label.
        for cluster_label in sorted(set(hemisphere_values) - {0}):

            # Identify the full-surface vertices belonging to this cluster.
            cluster_vertices = full_vertex_indices[
                hemisphere_values == cluster_label
            ]

            # Calculate the cluster centroid from surface coordinates.
            centroid = surface_coordinates[hemisphere][cluster_vertices].mean(axis=0)

            # Calculate the Euclidean distance from the manuscript pMTG center.
            distance = np.linalg.norm(
                centroid - PMTG_TARGET_COORDINATES[hemisphere]
            )

            # Store the cluster summary row.
            rows.append(
                {
                    "analysis": analysis_name,
                    "hemisphere": hemisphere,
                    "cluster_label": int(cluster_label),
                    "vertices": int(cluster_vertices.size),
                    "centroid_x": float(centroid[0]),
                    "centroid_y": float(centroid[1]),
                    "centroid_z": float(centroid[2]),
                    "distance_to_pmtg_target_mm": float(distance),
                }
            )

    # Return all cluster summaries as a table.
    return pd.DataFrame(rows)


def select_pmtg_labels(cluster_summary, analysis_name):
    """Select one left and one right pMTG label from a cluster-summary table."""

    selected = {}

    selected_rows = []

    # Select a pMTG label separately for left and right hemispheres.
    for hemisphere in ("left", "right"):

        # Keep only candidate clusters from the current hemisphere.
        hemisphere_summary = cluster_summary.loc[
            cluster_summary["hemisphere"].eq(hemisphere)
        ].copy()

        # Stop if Workbench found no cluster in the current hemisphere.

        # Sort candidates by distance from the expected pMTG center.
        sorted_candidates = hemisphere_summary.sort_values(
            "distance_to_pmtg_target_mm"
        )

        # Use the nearest candidate as the selected pMTG cluster.
        selected_row = sorted_candidates.iloc[0]

        # Record how the selected row was chosen.
        selection_method = "nearest manuscript center"

        # Store the selected integer label for later masking.
        selected[hemisphere] = int(selected_row["cluster_label"])

        # Convert the selected row to a mutable dictionary.
        selected_record = selected_row.to_dict()

        # Add the selection method to the selected-row record.
        selected_record["selection_method"] = selection_method

        # Store the selected-row record.
        selected_rows.append(selected_record)

    # Return selected labels and their summary table.
    return selected, pd.DataFrame(selected_rows)


def make_pmtg_masks(cluster_values, selected_labels):
    """Create left, right, and bilateral pMTG masks from selected cluster labels."""

    # Convert the cluster map to a one-dimensional array for Boolean masking.
    cluster_values = np.asarray(cluster_values).squeeze()

    # Stop if the cluster map does not match the template grayordinate axis length.

    # Stop if either required hemisphere label is missing.
    missing_hemispheres = [
        hemisphere for hemisphere in ("left", "right")
        if hemisphere not in selected_labels
    ]

    # Create one full-grayordinate Boolean mask per hemisphere.
    masks = {}

    # Build hemisphere-specific masks so duplicated Workbench labels cannot bleed across hemispheres.
    for hemisphere in ("left", "right"):

        # Retrieve this hemisphere's CIFTI slice and selected integer cluster label.
        hemisphere_slice = hemisphere_models[hemisphere]["slice"]
        selected_label = int(selected_labels[hemisphere])

        # Mark only vertices from the selected hemisphere and selected cluster label.
        mask = np.zeros(n_grayordinates, dtype=bool)
        mask[hemisphere_slice] = cluster_values[hemisphere_slice] == selected_label
        masks[hemisphere] = mask

    # Combine the left and right masks for bilateral Dice comparisons.
    masks["bilateral"] = np.logical_or(masks["left"], masks["right"])

    # Return the left, right, and bilateral pMTG masks.
    return masks


def dice_overlap_metrics(reference, comparison):
    """Calculate Dice coefficient and supporting vertex counts for two masks."""

    # Convert the reference input to a Boolean mask.
    reference_mask = np.asarray(reference, dtype=bool)

    # Convert the comparison input to a Boolean mask.
    comparison_mask = np.asarray(comparison, dtype=bool)

    # Stop if the masks do not have matching shapes.

    # Count vertices in the reference mask.
    reference_vertices = int(reference_mask.sum())

    # Count vertices in the comparison mask.
    comparison_vertices = int(comparison_mask.sum())

    # Count vertices shared by both masks.
    intersection_vertices = int(np.logical_and(reference_mask, comparison_mask).sum())

    # Treat two empty masks as perfectly overlapping.
    if reference_vertices + comparison_vertices == 0:
        dice = 1.0

    # Otherwise compute the standard Dice coefficient.
    else:
        dice = 2 * intersection_vertices / (reference_vertices + comparison_vertices)

    # Return Dice coefficient and supporting counts.
    return {
        "reference_vertices": reference_vertices,
        "comparison_vertices": comparison_vertices,
        "intersection_vertices": intersection_vertices,
        "dice": float(dice),
    }


def compare_pmtg_masks(first_masks, second_masks, first_name, second_name, analysis):
    """Compare left, right, and bilateral pMTG masks using Dice coefficient."""

    rows = []

    # Compare the left, right, and bilateral masks.
    for region in ("left", "right", "bilateral"):

        # Calculate Dice coefficient and supporting counts for this region.
        metrics = dice_overlap_metrics(first_masks[region], second_masks[region])

        # Store the comparison row.
        rows.append(
            {
                "analysis": analysis,
                "first_map": first_name,
                "second_map": second_name,
                "region": region,
                **metrics,
            }
        )

    # Return all comparison rows.
    return rows


## Subject-Level Thresholding and Clustering

Each participant map is thresholded at its participant-specific lower-tail percentile and then clustered with Workbench to retain components of at least 30 mm2.


In [ ]:
subject_variant_paths = {}

subject_threshold_rows = []

# Generate subject maps at each requested lower-tail threshold.
for percentile in VARIANT_PERCENTILES:

    # Create the output folder for this percentile.
    percentile_dir = SUBJECT_OUTPUT_DIR / f"bottom_{percentile:g}_percent"

    # Ensure the percentile-specific output folder exists.
    percentile_dir.mkdir(parents=True, exist_ok=True)

    # Create a nested dictionary for this percentile's subject maps.
    subject_variant_paths[percentile] = {}

    # Generate a thresholded-and-clustered map for each cohort participant.
    for subject in COHORT_SUBJECTS:

        # Retrieve this participant's unthresholded spatial-correlation map.
        spatial_path = spatial_correlation_index[subject]

        # Load this participant's spatial-correlation values.
        spatial_values = load_cifti_vector(spatial_path)

        # Calculate this participant's lower-tail percentile cutoff.
        cutoff = float(np.percentile(spatial_values, percentile))

        # Define the output path for this subject and percentile.
        output_path = (
            percentile_dir
            / f"{subject}_networkVariants_bottom{percentile:g}_min30mm2.dtseries.nii"
        )

        # Run Workbench lower-tail clustering at the subject-level area threshold.
        run_find_clusters(
            spatial_path,
            cutoff,
            SUBJECT_MIN_CLUSTER_AREA_MM2,
            output_path,
            less_than=True,
        )

        # Load the clustered map to count retained vertices.
        clustered_values = load_cifti_vector(output_path)

        # Store the clustered map path for later group consensus analyses.
        subject_variant_paths[percentile][subject] = output_path

        # Store this subject-threshold summary row.
        subject_threshold_rows.append(
            {
                "subject": subject,
                "percentile": percentile,
                "correlation_cutoff": cutoff,
                "target_vertices_before_clustering": int((spatial_values < cutoff).sum()),
                "retained_vertices_after_30mm2_clustering": int((clustered_values != 0).sum()),
                "cluster_map_path": str(output_path),
            }
        )

# Convert subject-threshold summary rows into a table.
subject_threshold_df = pd.DataFrame(subject_threshold_rows)

# Display the subject-threshold summary table.
display(subject_threshold_df)

# Save the subject-threshold summary table.
subject_threshold_df.to_csv(OUTPUT_DIR / 'subject_threshold_and_cluster_summary.csv', index=False)


## Full-Cohort Group Maps at 5%, 10%, and 15%

The full cohort is clustered at 5/10 consensus for each threshold. This section saves the group density maps, group cluster maps, and cluster summaries.


In [ ]:
# Create dictionaries and lists for full-cohort group-map results.
group_results = {}
group_pmtg_masks = {}
group_map_rows = []
group_candidate_rows = []
group_selection_rows = []

# Build one full-cohort consensus map for each requested threshold.
for percentile in VARIANT_PERCENTILES:

    # Name this full-cohort analysis by the lower-tail percentile.
    analysis_name = f"full_cohort_bottom{percentile:g}"

    # Build the full-cohort consensus map using a 5/10 rule.
    result = build_group_clusters(
        [
            subject_variant_paths[percentile][subject]
            for subject in COHORT_SUBJECTS
        ],
        analysis_name,
        min_subjects=5,
    )

    # Summarize all Workbench clusters in the full-cohort map.
    cluster_summary = summarize_clusters(result["cluster_values"], analysis_name)

    # Add threshold metadata to the candidate-cluster summary.
    cluster_summary["percentile"] = percentile
    cluster_summary["minimum_subject_count"] = result["min_subjects"]

    # Select the left and right pMTG labels from the full-cohort map.
    selected_labels, selected_summary = select_pmtg_labels(
        cluster_summary,
        analysis_name,
    )

    # Add threshold metadata to the selected-label summary.
    selected_summary["percentile"] = percentile
    selected_summary["minimum_subject_count"] = result["min_subjects"]

    # Create left, right, and bilateral pMTG masks for this threshold.
    pmtg_masks = make_pmtg_masks(result["cluster_values"], selected_labels)

    # Store the complete full-cohort result and selected pMTG masks.
    group_results[percentile] = result
    group_pmtg_masks[percentile] = pmtg_masks

    # Store output paths for this full-cohort threshold.
    group_map_rows.append(
        {
            "percentile": percentile,
            "minimum_subject_count": result["min_subjects"],
            "density_map": str(result["density_path"]),
            "cluster_map": str(result["cluster_path"]),
        }
    )

    # Store the cluster and selected-label summaries.
    group_candidate_rows.append(cluster_summary)
    group_selection_rows.append(selected_summary)

    # Create a two-label full-cohort pMTG map for Workbench inspection.
    labeled_pmtg = (
        pmtg_masks["left"].astype(np.int16)
        + 2 * pmtg_masks["right"].astype(np.int16)
    )

    # Save this selected full-cohort pMTG map.
    save_scalar_cifti(
        labeled_pmtg,
        GROUP_OUTPUT_DIR / f"{analysis_name}_pmtg.dscalar.nii",
        f"{analysis_name}_pmtg",
    )

# Convert full-cohort group-map output rows into a table.
group_map_df = pd.DataFrame(group_map_rows)

# Combine all full-cohort candidate-cluster summaries.
threshold_cluster_summary_df = pd.concat(
    group_candidate_rows,
    ignore_index=True,
).sort_values(
    ["percentile", "hemisphere", "distance_to_pmtg_target_mm"]
)

# Combine selected full-cohort pMTG label summaries.
threshold_selected_summary_df = pd.concat(
    group_selection_rows,
    ignore_index=True,
).sort_values(["percentile", "hemisphere"])

# Store the full-cohort 10% result and pMTG masks used by later analyses.
full_10_result = group_results[JACKKNIFE_PERCENTILE]
full_10_pmtg_masks = group_pmtg_masks[JACKKNIFE_PERCENTILE]
full_10_selected_summary_df = threshold_selected_summary_df.loc[
    threshold_selected_summary_df["percentile"].eq(JACKKNIFE_PERCENTILE)
].copy()

# Display full-cohort group-map outputs and selected pMTG labels.
display(group_map_df)
display(threshold_selected_summary_df.round(3))

# Save full-cohort group-map outputs and cluster summaries.
group_map_df.to_csv(OUTPUT_DIR / 'threshold_group_map_outputs.csv', index=False)
threshold_cluster_summary_df.to_csv(OUTPUT_DIR / 'threshold_group_cluster_candidates.csv', index=False)
threshold_selected_summary_df.to_csv(OUTPUT_DIR / 'threshold_selected_pmtg_labels.csv', index=False)
full_10_selected_summary_df.to_csv(OUTPUT_DIR / 'full_10percent_selected_pmtg_labels.csv', index=False)


## Diagnosis-Exclusion Comparison at 10%

The diagnosis-excluded map uses a 4/7 consensus rule and is compared with the full-cohort 10% map using Dice coefficient.


In [ ]:
# Name the diagnosis-excluded 10% analysis.
td_analysis_name = "td_only_bottom10"

# Build the diagnosis-excluded consensus map using a 4/7 rule.
td_result = build_group_clusters(
    [subject_variant_paths[10.0][subject] for subject in TD_SUBJECTS],
    td_analysis_name,
    min_subjects=4,
)

# Summarize all Workbench clusters in the diagnosis-excluded map.
td_cluster_summary_df = summarize_clusters(
    td_result["cluster_values"],
    td_analysis_name,
)

# Select the diagnosis-excluded left and right pMTG labels.
td_selected_labels, td_selected_summary_df = select_pmtg_labels(
    td_cluster_summary_df,
    td_analysis_name,
)

# Create diagnosis-excluded left, right, and bilateral pMTG masks.
td_pmtg_masks = make_pmtg_masks(
    td_result["cluster_values"],
    td_selected_labels,
)

# Display the selected diagnosis-excluded pMTG labels.
display(td_selected_summary_df.round(3))

# Compare the full-cohort 10% map with the diagnosis-excluded 10% map.
diagnosis_exclusion_df = pd.DataFrame(
    compare_pmtg_masks(
        full_10_pmtg_masks,
        td_pmtg_masks,
        "Full cohort 5/10",
        "Diagnosis-excluded 4/7",
        "Diagnosis exclusion at 10%",
    )
)

# Display the diagnosis-exclusion Dice comparison.
display(diagnosis_exclusion_df.round(4))

# Save the diagnosis-exclusion Dice comparison.
diagnosis_exclusion_df.to_csv(OUTPUT_DIR / 'diagnosis_exclusion_pmtg_dice.csv', index=False)

# Save all diagnosis-excluded candidate cluster summaries.
td_cluster_summary_df.to_csv(OUTPUT_DIR / 'diagnosis_exclusion_all_cluster_candidates.csv', index=False)

# Save selected diagnosis-excluded pMTG labels.
td_selected_summary_df.to_csv(OUTPUT_DIR / 'diagnosis_exclusion_selected_pmtg_labels.csv', index=False)

# Create a two-label diagnosis-excluded pMTG map.
td_labeled_pmtg = (
    td_pmtg_masks["left"].astype(np.int16)
    + 2 * td_pmtg_masks["right"].astype(np.int16)
)

# Save the diagnosis-excluded pMTG map for Workbench inspection.
save_scalar_cifti(
    td_labeled_pmtg,
    GROUP_OUTPUT_DIR / "td_only_bottom10_pmtg.dscalar.nii",
    "td_only_bottom10_pmtg",
)


## Ten-Percent Jackknife Analysis

Each jackknife map leaves out one participant from the full reported cohort, uses a 4/9 consensus rule, and is compared with the full-cohort 10% map and with every other jackknife map using bilateral Dice coefficient. Sensitivity to removing one participant is summarized from bilateral leave-one-out Dice mean, Dice loss, sample variance, standard deviation, and min/max values.


In [ ]:
# Create dictionaries and lists for jackknife results.
jackknife_results = {}
jackknife_pmtg_masks = {}
jackknife_selection_rows = []
jackknife_candidate_rows = []
jackknife_full_comparison_rows = []

# Build one 10% consensus map for each omitted participant in the full cohort.
for omitted_subject in COHORT_SUBJECTS:

    # Define the nine participants retained in this jackknife run.
    retained_subjects = [
        subject for subject in COHORT_SUBJECTS
        if subject != omitted_subject
    ]

    # Name this jackknife analysis by the omitted participant.
    analysis_name = f"jackknife_without_{omitted_subject}_bottom10"

    # Build the jackknife consensus map using the requested 4/9 rule.
    result = build_group_clusters(
        [
            subject_variant_paths[JACKKNIFE_PERCENTILE][subject]
            for subject in retained_subjects
        ],
        analysis_name,
        min_subjects=JACKKNIFE_MIN_SUBJECT_COUNT,
    )

    # Summarize all Workbench clusters in the jackknife map.
    cluster_summary = summarize_clusters(result["cluster_values"], analysis_name)

    # Select the left and right pMTG labels from the jackknife map.
    selected_labels, selected_summary = select_pmtg_labels(
        cluster_summary,
        analysis_name,
    )

    # Create left, right, and bilateral pMTG masks for this jackknife run.
    pmtg_masks = make_pmtg_masks(result["cluster_values"], selected_labels)

    # Store the complete jackknife result.
    jackknife_results[omitted_subject] = result

    # Store the selected jackknife pMTG masks.
    jackknife_pmtg_masks[omitted_subject] = pmtg_masks

    # Add the omitted participant to the selected-label summary.
    selected_summary["omitted_subject"] = omitted_subject

    # Add the jackknife consensus count to the selected-label summary.
    selected_summary["minimum_subject_count"] = result["min_subjects"]

    # Store the selected-label summary.
    jackknife_selection_rows.append(selected_summary)

    # Add the omitted participant to the candidate-cluster summary.
    cluster_summary["omitted_subject"] = omitted_subject

    # Store the candidate-cluster summary.
    jackknife_candidate_rows.append(cluster_summary)

    # Compare this jackknife pMTG map with the full-cohort 10% pMTG map.
    jackknife_full_comparison_rows.extend(
        compare_pmtg_masks(
            full_10_pmtg_masks,
            pmtg_masks,
            "Full cohort 10%",
            f"Without {omitted_subject}",
            "Jackknife versus full 10% map",
        )
    )

    # Create a two-label jackknife pMTG map for Workbench inspection.
    labeled_pmtg = (
        pmtg_masks["left"].astype(np.int16)
        + 2 * pmtg_masks["right"].astype(np.int16)
    )

    # Save this selected jackknife pMTG map.
    save_scalar_cifti(
        labeled_pmtg,
        GROUP_OUTPUT_DIR / f"{analysis_name}_pmtg.dscalar.nii",
        f"{analysis_name}_pmtg",
    )

# Combine selected jackknife label summaries.
jackknife_selection_df = pd.concat(
    jackknife_selection_rows,
    ignore_index=True,
)

# Combine candidate jackknife cluster summaries.
jackknife_candidate_df = pd.concat(
    jackknife_candidate_rows,
    ignore_index=True,
).sort_values(
    ["omitted_subject", "hemisphere", "distance_to_pmtg_target_mm"]
)

# Display selected jackknife pMTG labels.
display(jackknife_selection_df.round(3))

# Save selected jackknife pMTG labels.
jackknife_selection_df.to_csv(OUTPUT_DIR / 'jackknife_selected_pmtg_labels.csv', index=False)

# Save all jackknife candidate-cluster summaries.
jackknife_candidate_df.to_csv(OUTPUT_DIR / 'jackknife_all_cluster_candidates.csv', index=False)

# Convert jackknife-versus-full comparison rows into a table.
jackknife_vs_full_df = pd.DataFrame(jackknife_full_comparison_rows)

# Keep only bilateral jackknife-versus-full Dice comparisons for reporting.
bilateral_jackknife_vs_full_df = jackknife_vs_full_df.loc[
    jackknife_vs_full_df["region"].eq("bilateral")
].copy()

# Display bilateral jackknife-versus-full Dice comparisons.
display(bilateral_jackknife_vs_full_df.round(4))

# Save bilateral jackknife-versus-full Dice comparisons.
bilateral_jackknife_vs_full_df.to_csv(OUTPUT_DIR / 'jackknife_vs_full_bilateral_pmtg_dice.csv', index=False)

# Count the full-cohort jackknife resamples.
n_jackknife_subjects = len(COHORT_SUBJECTS)

# Keep only bilateral leave-one-out Dice values for jackknife sensitivity summaries.
jackknife_dice_df = bilateral_jackknife_vs_full_df.loc[
    :,
    ["analysis", "first_map", "second_map", "region", "dice"],
].copy()

# Recover the omitted participant label from the comparison table.
jackknife_dice_df["omitted_subject"] = jackknife_dice_df["second_map"].str.replace(
    "Without ",
    "",
    regex=False,
)

# Store Dice loss from perfect overlap for sensitivity ranking.
jackknife_dice_df["leave_one_out_dice_loss"] = 1.0 - jackknife_dice_df["dice"]

# Create summary rows for leave-one-out Dice sensitivity estimates.
jackknife_sensitivity_rows = []

# Summarize bilateral leave-one-out Dice values.
for region, group in jackknife_dice_df.groupby("region", sort=False):

    # Convert leave-one-out Dice values to floating point for jackknife formulas.
    leave_one_out_dice = group["dice"].to_numpy(dtype=float)

    # Calculate the mean leave-one-out Dice value.
    mean_leave_one_out_dice = float(leave_one_out_dice.mean())

    # Estimate sensitivity as the sample variance of leave-one-out Dice values.
    leave_one_out_dice_sample_variance = float(leave_one_out_dice.var(ddof=1))

    # Store the corresponding sample standard deviation.
    leave_one_out_dice_sample_std = float(np.sqrt(leave_one_out_dice_sample_variance))

    # Store a compact Dice-only summary row for this region.
    jackknife_sensitivity_rows.append(
        {
            "region": region,
            "jackknife_samples": n_jackknife_subjects,
            "mean_leave_one_out_dice": mean_leave_one_out_dice,
            "mean_leave_one_out_dice_loss": float(
                group["leave_one_out_dice_loss"].mean()
            ),
            "maximum_leave_one_out_dice_loss": float(
                group["leave_one_out_dice_loss"].max()
            ),
            "minimum_leave_one_out_dice": float(leave_one_out_dice.min()),
            "maximum_leave_one_out_dice": float(leave_one_out_dice.max()),
            "leave_one_out_dice_sample_variance": leave_one_out_dice_sample_variance,
            "leave_one_out_dice_sample_standard_deviation": leave_one_out_dice_sample_std,
            "leave_one_out_dice_standard_error_of_mean": float(
                leave_one_out_dice_sample_std / np.sqrt(n_jackknife_subjects)
            ),
        }
    )

# Convert jackknife Dice sensitivity summaries into a table.
jackknife_sensitivity_df = pd.DataFrame(jackknife_sensitivity_rows)

# Display bilateral leave-one-out Dice values used for jackknife sensitivity summaries.
display(jackknife_dice_df.round(4))

# Display the Dice-only jackknife sensitivity summary.
display(jackknife_sensitivity_df.round(4))

# Save bilateral leave-one-out Dice values used for jackknife sensitivity summaries.
jackknife_dice_df.to_csv(OUTPUT_DIR / 'jackknife_leave_one_out_bilateral_dice.csv', index=False)

# Save the Dice-only jackknife sensitivity summary.
jackknife_sensitivity_df.to_csv(OUTPUT_DIR / 'jackknife_dice_sensitivity_summary.csv', index=False)

jackknife_pairwise_rows = []

# Compare every unique pair of jackknife pMTG maps.
for first_index, first_subject in enumerate(COHORT_SUBJECTS):

    # Compare the current omitted participant with every later omitted participant.
    for second_subject in COHORT_SUBJECTS[first_index + 1:]:

        # Add left, right, and bilateral Dice rows for this jackknife pair.
        jackknife_pairwise_rows.extend(
            compare_pmtg_masks(
                jackknife_pmtg_masks[first_subject],
                jackknife_pmtg_masks[second_subject],
                f"Without {first_subject}",
                f"Without {second_subject}",
                "Pairwise jackknife comparison",
            )
        )

# Convert pairwise jackknife Dice rows into a table.
jackknife_pairwise_df = pd.DataFrame(jackknife_pairwise_rows)

# Keep only bilateral pairwise jackknife Dice comparisons for reporting.
bilateral_pairwise_df = jackknife_pairwise_df.loc[
    jackknife_pairwise_df["region"].eq("bilateral")
].copy()

# Display bilateral pairwise jackknife Dice comparisons.
display(bilateral_pairwise_df.round(4))

# Save bilateral pairwise jackknife Dice comparisons.
bilateral_pairwise_df.to_csv(OUTPUT_DIR / 'jackknife_pairwise_bilateral_pmtg_dice.csv', index=False)

# Create an identity matrix for bilateral jackknife Dice values.
dice_matrix = pd.DataFrame(
    np.eye(len(COHORT_SUBJECTS)),
    index=COHORT_SUBJECTS,
    columns=COHORT_SUBJECTS,
)

# Fill the Dice matrix with pairwise jackknife values.
for row in bilateral_pairwise_df.itertuples():

    # Extract the first omitted participant name from the table label.
    first_subject = row.first_map.replace("Without ", "")

    # Extract the second omitted participant name from the table label.
    second_subject = row.second_map.replace("Without ", "")

    # Store the Dice value in the upper triangle.
    dice_matrix.loc[first_subject, second_subject] = row.dice

    # Store the same Dice value in the lower triangle.
    dice_matrix.loc[second_subject, first_subject] = row.dice

# Create the bilateral jackknife Dice heatmap.
figure, axis = plt.subplots(figsize=(8, 7))

# Draw the heatmap with Dice values printed in each cell.
sns.heatmap(
    dice_matrix,
    vmin=0,
    vmax=1,
    cmap="viridis",
    square=True,
    annot=True,
    fmt=".2f",
    cbar_kws={"label": "Dice coefficient"},
    ax=axis,
)

# Label the jackknife heatmap.
axis.set_title("Bilateral pMTG Jackknife Dice")

# Label the x-axis by omitted participant.
axis.set_xlabel("Omitted participant")

# Label the y-axis by omitted participant.
axis.set_ylabel("Omitted participant")

# Improve figure spacing.
figure.tight_layout()

# Save the jackknife Dice heatmap.
figure.savefig(
    OUTPUT_DIR / "jackknife_pairwise_bilateral_dice.png",
    dpi=300,
)

# Show the jackknife Dice heatmap.
plt.show()


## Output Summary

The final cell lists the generated group maps, prints the separate diagnosis-exclusion Dice value, and summarizes whole-cohort bilateral jackknife Dice sensitivity to removing one participant at a time.


In [ ]:
# Print the generated full-cohort group maps.
print("Full-cohort group maps:")

# Print one density-map and one cluster-map path for each threshold.
for row in group_map_df.itertuples():

    # Print the density-map path for this threshold.
    print(f"  Bottom {row.percentile:g}% density map: {row.density_map}")

    # Print the cluster-map path for this threshold.
    print(f"  Bottom {row.percentile:g}% cluster map: {row.cluster_map}")

# Select the bilateral diagnosis-exclusion comparison row.
bilateral_exclusion = diagnosis_exclusion_df.loc[
    diagnosis_exclusion_df["region"].eq("bilateral")
].iloc[0]

# Print the bilateral diagnosis-exclusion summary from the separate comparison.
print(f"Diagnosis-exclusion bilateral pMTG: Dice={bilateral_exclusion['dice']:.3f}")

# Reuse the bilateral pairwise jackknife table.
bilateral_jackknife = bilateral_pairwise_df

# Print the bilateral pairwise jackknife Dice range.
print(
    "Pairwise bilateral 10% jackknife overlap: "
    f"Dice range={bilateral_jackknife['dice'].min():.3f}-"
    f"{bilateral_jackknife['dice'].max():.3f}."
)

# Select the bilateral whole-cohort jackknife Dice sensitivity summary.
bilateral_jackknife_dice = jackknife_sensitivity_df.loc[
    jackknife_sensitivity_df["region"].eq("bilateral")
].iloc[0]

# Print the bilateral Dice jackknife sensitivity summary.
print(
    "Whole-cohort jackknife bilateral Dice versus full map: "
    f"mean={bilateral_jackknife_dice['mean_leave_one_out_dice']:.3f}, "
    f"sample variance={bilateral_jackknife_dice['leave_one_out_dice_sample_variance']:.5f}, "
    f"sample SD={bilateral_jackknife_dice['leave_one_out_dice_sample_standard_deviation']:.3f}, "
    f"SE mean={bilateral_jackknife_dice['leave_one_out_dice_standard_error_of_mean']:.3f}, "
    f"mean loss={bilateral_jackknife_dice['mean_leave_one_out_dice_loss']:.3f}."
)
